In [29]:
import re

In [30]:
def load_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        return file.read()


In [71]:
def get_annotation_entities(annotations, select_types=None):
    entities = {}
    lines = annotations.split('\n')
    for line in lines:
        if line.startswith('T'):
            parts = line.split('\t')
            if len(parts) < 3:
                continue # Ignore lines that do not have the required parts
            entity_id = parts[0]
            term_info = parts[1].split()
            if select_types and term_info[0] not in select_types:
                continue # Filter out unwanted types if specified
            entity_text = parts[2]
            position_data = term_info[1] # This should be something like '26 40;44 70'
            spans = []
            # Splitting at each space, then each pair is separated by semicolon
            for pos_pair in position_data.split(';'):
                start_end = pos_pair.split()
                if len(start_end) == 2: # Ensure there is a start and an end
                    start, end = int(start_end[0]), int(start_end[1])
                    if end > start: # Ensure the end is greater than the start
                        spans.append((start, end))
            if spans:
                entities[entity_id] = {'spans': spans, 'type': term_info[0]}
    return entities

def parse_annotations(annotations):
    relations = []
    entities = get_annotation_entities(annotations)
    lines = annotations.split('\n')
    for line in lines:
        if line.startswith('R') or line.startswith('*'):
            parts = line.split('\t')
            if len(parts) < 2:
                continue # Skip malformed lines
            relation_details = parts[1].split(' ')
            if len(relation_details) < 3:
                continue # Skip lines that do not have the required parts for relations
            relation_type = relation_details[0]
            arg1 = relation_details[1].split(':')[1] if ':' in relation_details[1] else None
            arg2 = relation_details[2].split(':')[1] if ':' in relation_details[2] else None
            if arg1 and arg2 and relation_type in ['OR', 'AND']:
                relations.append({'type': relation_type, 'arg1': arg1, 'arg2': arg2})
    return relations, entities

def mark_relations_in_text(text, relations, entities):
    marked_text = text
    for relation in relations:
        if relation['type'] in ['AND', 'OR']:
            entity1 = entities.get(relation['arg1'])
            entity2 = entities.get(relation['arg2'])
            if entity1 and entity2:
                # Finden Sie die Positionen der ersten und letzten Wörter der beiden Entitäten
                start1, end1 = entity1['spans'][0]
                start2, end2 = entity2['spans'][-1]

                # Markieren Sie den Text zwischen den Positionen mit den entsprechenden Beziehungen
                marked_text = marked_text[:start1] + f"[{relation['type']}]" + marked_text[start1:end2] + f"[/{relation['type']}]" + marked_text[end2:]
    return marked_text

def load_file(filepath):
    with open(filepath, 'r') as file:
        return file.read()

def main(txt_filepath, ann_filepath):
    text = load_file(txt_filepath)
    annotations = load_file(ann_filepath)
    relations, entities = parse_annotations(annotations)
    marked_text = mark_relations_in_text(text, relations, entities)
    print(marked_text)


In [72]:
ann_file = 'NCT00050349_exc.ann'
main('NCT00050349_exc.txt', ann_file)

Patients with symptomatic CNS metastases or leptomeningeal involvement 
Patients with known brain metastases, unless these metastases have been treated and/or have been stable for at least six months prior to study start. Subjects with a history of brain metastases must have a head CT with contrast to document either response or progression. 
Patients with bone metastases as the only site(s) of measurable disease 
Patients with hepatic artery chemoembolization within the last 6 months (one month if there are other sites of measurable disease) 
Patients who have been previously treated with radioactive directed therapies 
Patients who have been previously treated with epothilone 
Patients with any peripheral neuropathy or unresolved diarrhea greater than Grade 1 
Patients with severe cardiac insufficiency patients taking Coumadin or other warfarin-containing agents with the exception of low dose warfarin (1 mg or less) for the maintenance of in-dwelling lines or ports 
Patients taking a

In [84]:
def extract_and_or_relations_with_offsets(annotations):
    relations_with_offsets = []
    current_line = ""
    for line in annotations.split('\n'):
        if line.startswith('R') or line.startswith('*'):
            current_line += line + "\n"
        elif line.startswith('T'):
            current_line += line + "\n"

        if "AND" in current_line or "OR" in current_line:
            relation_type = "AND" if "AND" in current_line else "OR"
            entities = [entity.strip() for entity in current_line.split('\n') if entity.startswith('T')]

            # Überprüfe, ob die Entitäten die erwartete Struktur haben
            if all([entity.split('\t') for entity in entities]):
                # Stelle sicher, dass die split-Methode die erwartete Anzahl von Teilen erzeugt
                offsets = [(entity.split('\t')[2], entity.split('\t')[1]) for entity in entities if len(entity.split('\t')) >= 3]
                relations_with_offsets.append({
                    'type': relation_type,
                    'entities': entities,
                    'offsets': offsets
                })
            else:
                print("Fehler: Die Entitäten haben keine erwartete Struktur.")

            current_line = ""

    return relations_with_offsets

# Beispielverwendung
annotations = load_file(ann_file)

relations_with_offsets = extract_and_or_relations_with_offsets(annotations)



In [85]:
relations_with_offsets

[{'type': 'OR',
  'entities': ['T1\tCondition 26 40\tCNS metastases',
   'T2\tCondition 44 70\tleptomeningeal involvement'],
  'offsets': [('CNS metastases', 'Condition 26 40'),
   ('leptomeningeal involvement', 'Condition 44 70')]},
 {'type': 'OR',
  'entities': ['T5\tProcedure 144 151\ttreated',
   'T6\tQualifier 164 179\tbeen stable for',
   'T7\tTemporal 180 220\tat least six months prior to study start'],
  'offsets': [('treated', 'Procedure 144 151'),
   ('been stable for', 'Qualifier 164 179'),
   ('at least six months prior to study start', 'Temporal 180 220')]},
 {'type': 'AND',
  'entities': ['T4\tCondition 92 108\tbrain metastases',
   'T8\tObservation 238 248\thistory of',
   'T9\tCondition 249 265\tbrain metastases'],
  'offsets': [('brain metastases', 'Condition 92 108'),
   ('history of', 'Observation 238 248'),
   ('brain metastases', 'Condition 249 265')]},
 {'type': 'AND',
  'entities': ['T10\tProcedure 278 299\thead CT with contrast',
   'T11\tScope 238 265\thistory 